## Encontro 2 - Validação temporal e métricas


## Importar e organizar os dados

In [1]:
import pandas as pd

df = pd.read_csv("../dados/mecaniqa_dataset.csv")
df.columns = df.columns.str.strip()
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index().asfreq("D")

colunas = ["Trocas_Oleo", "Manutencao_Motor"]
df = df[colunas]

df.head()

,Trocas_Oleo,Manutencao_Motor
Data,,
2024-01-01,11.0,2.0
2024-01-02,9.0,4.0
2024-01-03,12.0,1.0
2024-01-04,15.0,8.0
2024-01-05,25.0,10.0


## Tratar os dados

In [2]:
df["Trocas_Oleo"] = df["Trocas_Oleo"].clip(upper=55)
df["Manutencao_Motor"] = df["Manutencao_Motor"].clip(lower=0, upper=26)

# Preserva os alvos ausentes antes de preencher o histórico.
observado = df.copy()
df = df.ffill()

df.isna().sum()

Trocas_Oleo         0
Manutencao_Motor    0
dtype: int64

## Criar as previsões dos baselines


In [3]:
def modelo_naive(serie):
    return serie.shift(1)

def modelo_media_movel(serie, janela):
    return serie.shift(1).rolling(janela).mean()

previsoes = {}

for coluna in colunas:
    previsoes[coluna] = pd.DataFrame({
        "Real": observado[coluna],
        "Naive": modelo_naive(df[coluna]),
        "Media_7": modelo_media_movel(df[coluna], 7),
        "Media_30": modelo_media_movel(df[coluna], 30)
    })

## Separar os períodos de treino e teste


In [4]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
)

tscv = TimeSeriesSplit(n_splits=5)

for treino, teste in tscv.split(df):
    print(f"Treino até {df.index[treino[-1]].date()} | "
          f"Teste: {df.index[teste[0]].date()} a {df.index[teste[-1]].date()}")

Treino até 2024-05-05 | Teste: 2024-05-06 a 2024-09-03
Treino até 2024-09-03 | Teste: 2024-09-04 a 2025-01-02
Treino até 2025-01-02 | Teste: 2025-01-03 a 2025-05-03
Treino até 2025-05-03 | Teste: 2025-05-04 a 2025-09-01
Treino até 2025-09-01 | Teste: 2025-09-02 a 2025-12-31


## Avaliar cada período e calcular a média


In [5]:
def avaliar_baseline(tabela, modelo):
    erros_mae = []
    erros_rmse = []
    erros_mape = []

    for treino, teste in tscv.split(tabela):
        dados_teste = tabela.iloc[teste].dropna()
        real = dados_teste["Real"]
        previsao = dados_teste[modelo]

        erros_mae.append(mean_absolute_error(real, previsao))
        erros_rmse.append(np.sqrt(mean_squared_error(real, previsao)))

        # Exclui os zeros somente do MAPE.
        validos = real != 0
        if validos.any():
            erros_mape.append(
                mean_absolute_percentage_error(real[validos], previsao[validos]) * 100
            )

    mae = np.mean(erros_mae)
    rmse = np.mean(erros_rmse)
    mape = np.mean(erros_mape) if erros_mape else float("nan")

    return mae, rmse, mape

## Mostrar os resultados


In [6]:
resultados = []

for coluna in colunas:
    for modelo in ["Naive", "Media_7", "Media_30"]:
        mae, rmse, mape = avaliar_baseline(previsoes[coluna], modelo)
        resultados.append([coluna, modelo, mae, rmse, mape])

        print(f"\n{coluna} - {modelo}")
        print(f"Resultados do Baseline - MAE: {mae:.2f}, "
              f"RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")


Trocas_Oleo - Naive
Resultados do Baseline - MAE: 6.65, RMSE: 8.92, MAPE: 32.64%

Trocas_Oleo - Media_7
Resultados do Baseline - MAE: 7.36, RMSE: 8.09, MAPE: 37.22%

Trocas_Oleo - Media_30
Resultados do Baseline - MAE: 7.34, RMSE: 8.15, MAPE: 37.31%

Manutencao_Motor - Naive
Resultados do Baseline - MAE: 3.43, RMSE: 4.46, MAPE: 43.55%

Manutencao_Motor - Media_7
Resultados do Baseline - MAE: 3.25, RMSE: 3.87, MAPE: 44.30%

Manutencao_Motor - Media_30
Resultados do Baseline - MAE: 3.21, RMSE: 3.82, MAPE: 43.80%


## Comparar pelo menor MAE


In [7]:
resumo = pd.DataFrame(
    resultados, columns=["Serie", "Modelo", "MAE", "RMSE", "MAPE"]
)

resumo = resumo.sort_values(["Serie", "MAE"])

resumo.round(2)

,Serie,Modelo,MAE,RMSE,MAPE
5,Manutencao_Motor,Media_30,3.21,3.82,43.80
4,Manutencao_Motor,Media_7,3.25,3.87,44.30
3,Manutencao_Motor,Naive,3.43,4.46,43.55
0,Trocas_Oleo,Naive,6.65,8.92,32.64
2,Trocas_Oleo,Media_30,7.34,8.15,37.31
1,Trocas_Oleo,Media_7,7.36,8.09,37.22
